In [ ]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
COMMIT = '5bb03b18943e7c45a305ce1d78b3446c31829ae7'
ROOT = Path('/content/e07_source')
SETUP_ENV = dict(os.environ, UV_PYTHON_INSTALL_DIR='/content/e07_python', UV_CACHE_DIR='/content/e07_uv_cache', UV_NATIVE_TLS='true', CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='1', MPLCONFIGDIR='/content/e07_mpl', MPLBACKEND='Agg')
print(subprocess.check_output(['nvidia-smi'], text=True))
assert not ROOT.exists()
subprocess.run(['git','clone','--quiet','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(ROOT)],check=True)
subprocess.run(['git','checkout','--quiet','--detach',COMMIT],cwd=ROOT,check=True)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip()
source = {p.relative_to(ROOT).as_posix():p.read_text(encoding='utf-8') for folder in ('src','configs','experiments/configs') for p in (ROOT/folder).rglob('*') if p.suffix in {'.py','.json'}}
code_hash = hashlib.sha256(json.dumps(source,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
assert code_hash == 'cdea42adb930904fb0cefcd282d784bb3ac8caa311d3f1e12ef568622f9c12d6'
subprocess.run([sys.executable,'-m','pip','install','--target','/content/e07_tools','uv==0.12.7'],env=SETUP_ENV,check=True)
UV = Path('/content/e07_tools/bin/uv')
subprocess.run([str(UV),'python','install','3.11.14'],env=SETUP_ENV,check=True)
subprocess.run([str(UV),'venv','--seed','--python','3.11.14','/content/e07_venv'],env=SETUP_ENV,check=True)
PY = Path('/content/e07_venv/bin/python')
freeze = (ROOT/'docs/qualifications/evidence/rayyan_requirements_freeze.txt').read_text().splitlines()
lock = Path('/content/e07_lock.txt')
lock.write_text('\n'.join(x for x in freeze if not x.startswith('-e '))+'\n')
r = subprocess.run([str(UV),'pip','install','--python',str(PY),'torch==2.5.1+cu124','--index-url','https://download.pytorch.org/whl/cu124'],env=SETUP_ENV,capture_output=True,text=True); print(r.stdout,r.stderr[-1500:]); r.check_returncode()
r = subprocess.run([str(UV),'pip','install','--python',str(PY),'-r',str(lock)],env=SETUP_ENV,capture_output=True,text=True); print(r.stdout,r.stderr[-1500:]); r.check_returncode()
subprocess.run([str(UV),'pip','install','--python',str(PY),'-e',str(ROOT),'--no-deps'],env=SETUP_ENV,check=True)
subprocess.run([str(PY),'-m','pip','check'],env=SETUP_ENV,check=True)
print('E07 isolated environment provisioned:',COMMIT,code_hash,flush=True)